In [1]:
import os
import re
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [2]:
def load_corpus(data_dir):
    texts = []
    all_files = []
    for root, dirs, files in os.walk(data_dir):
        for fname in files:
            if fname.endswith('.txt'):
                all_files.append(os.path.join(root, fname))
    
    for fpath in tqdm(all_files, desc="Loading corpus"):
        with open(fpath, 'r', encoding='utf-8') as f:
            text = f.read().strip()
            if text: texts.append(text)
    return texts

train_dir = r"d:\dut_ai\AIO_code\Dence Representation\data\data_train\train"
corpus = load_corpus(train_dir)
print(f"Số documents: {len(corpus)}")

Loading corpus: 100%|██████████| 30000/30000 [08:14<00:00, 60.62it/s]

Số documents: 30000


In [3]:
def preprocess(texts, min_freq=3):
    all_tokens = []
    tokenized_docs = []
    for text in tqdm(texts, desc="Preprocessing"):
        text = text.lower()
        tokens = re.findall(r'[a-záàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵđ_]+', text)
        tokenized_docs.append(tokens)
        all_tokens.extend(tokens)
    
    freq = Counter(all_tokens)
    vocab = ['<UNK>'] + [w for w, c in freq.items() if c >= min_freq]
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}
    
    return tokenized_docs, vocab, word2idx, idx2word

tokenized_docs, vocab, word2idx, idx2word = preprocess(corpus, min_freq=3)
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

Preprocessing: 100%|██████████| 30000/30000 [00:01<00:00, 21396.10it/s]


Vocab size: 13676


In [4]:
def generate_cbow_data(tokenized_docs, word2idx, window=2):
    data = []
    for tokens in tqdm(tokenized_docs, desc="Generating CBOW pairs"):
        ids = [word2idx[w] if w in word2idx else word2idx['<UNK>'] for w in tokens]
        for i in range(window, len(ids) - window):
            target = ids[i]
            context = ids[i-window:i] + ids[i+1:i+window+1]
            data.append((context, target))
    return data

data = generate_cbow_data(tokenized_docs, word2idx, window=2)
print(f"Số cặp CBOW: {len(data)}")

Generating CBOW pairs: 100%|██████████| 30000/30000 [00:05<00:00, 5654.56it/s]

Số cặp CBOW: 2346007


In [5]:
class CBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super(CBOW, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear    = nn.Linear(embed_dim, vocab_size)
        
    def forward(self, context_ids):
        embeds     = self.embedding(context_ids)   # (batch, 4, embed_dim)
        mean_embed = embeds.mean(dim=1)            # (batch, embed_dim)
        out        = self.linear(mean_embed)       # (batch, vocab_size)
        return out

In [6]:
EMBED_DIM = 100
EPOCHS = 5
LR = 0.001
BATCH_SIZE = 1024
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Để tránh lỗi RAM, dùng subset
SUBSET_SIZE = min(len(data), 1000000)
subset_data = data[:SUBSET_SIZE]
train_data, val_data = train_test_split(subset_data, test_size=0.2, random_state=42)

def create_loader(data_list, batch_size):
    X = torch.tensor([ctx for ctx, target in data_list], dtype=torch.long)
    y = torch.tensor([target for ctx, target in data_list], dtype=torch.long)
    ds = torch.utils.data.TensorDataset(X, y)
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True)

train_loader = create_loader(train_data, BATCH_SIZE)
val_loader = create_loader(val_data, BATCH_SIZE)

model = CBOW(vocab_size, EMBED_DIM).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for xb, yb in pbar:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * xb.size(0)
        _, preds = torch.max(out, 1)
        train_correct += (preds == yb).sum().item()
        train_total += yb.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            val_loss += loss.item() * xb.size(0)
            _, preds = torch.max(out, 1)
            val_correct += (preds == yb).sum().item()
            val_total += yb.size(0)
    
    print(f"Epoch {epoch+1}: Train Loss: {train_loss/train_total:.4f}, Train Acc: {train_correct/train_total:.4f} | "
          f"Val Loss: {val_loss/val_total:.4f}, Val Acc: {val_correct/val_total:.4f}")

Epoch 1/5: 100%|██████████| 782/782 [02:27<00:00,  5.32it/s, loss=6.2878]


Epoch 1: Train Loss: 6.9959, Train Acc: 0.0421 | Val Loss: 6.2299, Val Acc: 0.0619


Epoch 2/5: 100%|██████████| 782/782 [03:32<00:00,  3.68it/s, loss=5.7778]


Epoch 2: Train Loss: 5.9999, Train Acc: 0.0724 | Val Loss: 5.9360, Val Acc: 0.0789


Epoch 3/5: 100%|██████████| 782/782 [05:10<00:00,  2.52it/s, loss=5.7426]


Epoch 3: Train Loss: 5.7176, Train Acc: 0.0872 | Val Loss: 5.7792, Val Acc: 0.0886


Epoch 4/5: 100%|██████████| 782/782 [05:10<00:00,  2.52it/s, loss=5.6291]


Epoch 4: Train Loss: 5.5321, Train Acc: 0.0970 | Val Loss: 5.6834, Val Acc: 0.0955


Epoch 5/5: 100%|██████████| 782/782 [05:07<00:00,  2.54it/s, loss=5.3326]


Epoch 5: Train Loss: 5.3961, Train Acc: 0.1045 | Val Loss: 5.6211, Val Acc: 0.1008


In [7]:
embedding_matrix = model.embedding.weight.data.cpu()
print(f"Embedding matrix shape: {embedding_matrix.shape}")
torch.save(embedding_matrix, "cbow_embeddings.pt")
print("Đã lưu embedding vào file cbow_embeddings.pt")

Embedding matrix shape: torch.Size([13676, 100])
Đã lưu embedding vào file cbow_embeddings.pt
